<a href="https://colab.research.google.com/github/isaaccg88/PredictiveMaintenance-AI-Activity5/blob/main/fuentes/entrena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Predictive Maintenance AI — Actividad 5
**Entrenamiento, validación cruzada y MLflow.** Notebook de ejecución y exploración; el experimento completo está implementado en `fuentes/train.py`. No se publican métricas simuladas.

## 1. Preparación del entorno
Se detecta si el notebook está ejecutándose en Colab o en Windows/Jupyter local. **Para obtener capturas en el panel de tu Windows, ejecuta el script desde Windows**: Colab no puede comunicarse con el `localhost` de tu computadora.

In [ ]:
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
print("Python:",sys.version.split()[0], "| Colab:",IN_COLAB)
print("Directorio actual:",Path.cwd())


## 2. Acceso al repositorio y dependencias
En Colab se clona la rama `main` y se instalan dependencias en el runtime. En Windows/Jupyter el notebook se ejecuta desde una copia clonada del repositorio.

In [ ]:
REPO_NAME = "PredictiveMaintenance-AI-Activity5"
if IN_COLAB:
    ROOT = Path("/content") / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/isaaccg88/" + REPO_NAME + ".git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT/"requirements.txt")], check=True)
else:
    cwd = Path.cwd().resolve()
    ROOT = next((p for p in [cwd, *cwd.parents] if (p/"fuentes"/"train.py").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Abre Jupyter desde la carpeta local del repositorio.")
os.chdir(ROOT)
print("Raíz del proyecto:", ROOT)


## 3. Calidad de datos, control de versiones y particiones
Se conserva el dataset original sin cambios y se genera uno limpio con `indice_original`, hash SHA256 y reporte de nulos/duplicados. No se aprenden transformaciones de prueba.

In [ ]:
from fuentes.datos_prep import prepare_data
data, splits, quality = prepare_data()
print("Registros:", len(data), "| Distribución:",quality["clases"])
print("Nulos:",quality["nulos_por_variable"])
print("Duplicados exactos:",quality["duplicados_exactos"])
print("Hash original:",quality["sha256_origen"])


## 4. Análisis exploratorio
Las visualizaciones 1–4 muestran desbalance, nulos, patrones por clase y tasa de falla por tipo. Interpretar sus implicaciones en README/PDF, sin confundir correlación con causalidad.

In [ ]:
from IPython.display import display, Image
for name in ["01_balance_clases.png","02_nulos.png","03_distribuciones.png","04_tipos_falla.png"]:
    display(Image(filename=str(ROOT/"resultados"/"figuras"/name)))


## 5. Modelos e hiperparámetros
Regresión logística y Random Forest, con transformers dentro del pipeline; validación cruzada estratificada de cinco folds y criterio Average Precision. Se prueban varias combinaciones Grid Search y se registran como runs hijos.

In [ ]:
from fuentes.train import pipeline_and_grid
for nombre in ("LogisticRegression","RandomForest"):
    _,grid = pipeline_and_grid(nombre)
    print(nombre, "rejilla:",grid)


## 6. Conexión con MLflow
**Local Windows:** `http://127.0.0.1:5000` con el servidor ya iniciado en PowerShell. **Colab:** SQLite propio del runtime, que no es el servidor de Windows y requiere exportar las evidencias. El conjunto de prueba sigue separado.

In [ ]:
if IN_COLAB:
    TRACKING_URI = "sqlite:////content/pm_actividad5_mlflow.db"
else:
    TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
print("Tracking URI:",TRACKING_URI)


## 7. Ejecutar entrenamiento completo y registro automático
Este paso ajusta ambas rejillas, crea experimentos MLflow, registra parámetros, modelos, artefactos y resultados; **elige con validación antes de abrir prueba**. Puede tardar varios minutos en CPU. No interrumpir mientras procesa.

In [ ]:
from fuentes.train import run
comparison = run(TRACKING_URI)
display(comparison.round(4))


## 8. Revisión de métricas y figuras
Comparar precisión, recall, F1, F2, ROC-AUC, Average Precision y coste computacional. Evitar atribuir significancia estadística a diferencias pequeñas sin incertidumbre.

In [ ]:
import pandas as pd
results = pd.read_csv(ROOT/"resultados"/"comparacion_modelos.csv")
display(results.round(4))
for image_name in ["05_06_comparacion_modelos.png","07_08_curvas_pr_roc.png","09_matrices_confusion_test.png"]:
    display(Image(filename=str(ROOT/"resultados"/"figuras"/image_name)))


## 9. Exportación verificable de MLflow
Exporta runs padre/hijo con parámetros y métricas; las **capturas reales** de MLflow para entrega deben guardarse aparte desde el panel del Windows. No se debe subir `mlflow.db` ni secretos al repositorio.

In [ ]:
from fuentes.export_evidencias import export_runs
runs_exportados = export_runs(TRACKING_URI)
display(runs_exportados.head(12))
print("RECUERDA: capturas del panel Windows y PDF académico pendientes de incorporar.")


## 10. Cierre y requisitos de la entrega
Después de verificar runs reales, exportar figuras/tablas y capturas, actualizar README, CHANGELOG y generar el **reporte técnico PDF** con interpretación cuantitativa y declaración de uso de ChatGPT. La entrega será únicamente la URL de GitHub, que contendrá el PDF y todas las evidencias.